In [1]:
from pyspark.sql import SparkSession

# Initialize SparkSession for Lab 5
spark = SparkSession.builder \
    .appName("Lab5-Flights-Streaming") \
    .getOrCreate()

# Set shuffle partitions to a lower number for optimal local performance
spark.conf.set("spark.sql.shuffle.partitions", "2")

# Read a static sample of the CSV to explore the actual columns
# Using file:// to enforce reading from the local container workspace
csv_sample_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("file:///data/spark-assignments/lab5_flights/csv_input_dir/*.csv")

print("📋 Actual CSV Schema:")
csv_sample_df.printSchema()

print("👀 Sample Data Rows:")
csv_sample_df.show(5)

📋 Actual CSV Schema:
root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: integer (nullable = true)

👀 Sample Data Rows:
+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows



In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Define the directory path inside the container workspace
CSV_INPUT_DIR = "file:///data/spark-assignments/lab5_flights/csv_input_dir"

# 2. Define the explicit schema based on our data exploration
flight_schema = StructType([
    StructField("DEST_COUNTRY_NAME", StringType(), True),
    StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
    StructField("count", IntegerType(), True)
])

# 3. Create the inbound Read Stream DataFrame
flight_stream_df = spark.readStream \
    .option("header", "true") \
    .schema(flight_schema) \
    .csv(CSV_INPUT_DIR)

# 4. Print schema to verify everything is mapped correctly
print("📊 Inbound CSV Stream Schema successfully initialized:")
flight_stream_df.printSchema()

📊 Inbound CSV Stream Schema successfully initialized:
root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: integer (nullable = true)



In [3]:
from pyspark.sql.functions import col, sum

# Aggregate the data at the level of origin and destination countries
# and calculate the total sum of flights for each pair
aggregated_flights_df = flight_stream_df \
    .groupBy("ORIGIN_COUNTRY_NAME", "DEST_COUNTRY_NAME") \
    .agg(sum(col("count")).alias("total_flights"))

# Print schema to see the structure of the aggregated DataFrame
print("📐 Aggregated Flights Data Schema Ready:")
aggregated_flights_df.printSchema()

📐 Aggregated Flights Data Schema Ready:
root
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- total_flights: long (nullable = true)



In [5]:
print("🚀 Starting Flights Stream Processing using Memory Sink...")

# Write the streaming output into an in-memory table named 'flights_view'
flight_query = aggregated_flights_df.writeStream \
    .format("memory") \
    .queryName("flights_view") \
    .outputMode("complete") \
    .start()

🚀 Starting Flights Stream Processing using Memory Sink...


In [6]:
# Query the in-memory table to see the aggregated results
spark.sql("SELECT * FROM flights_view ORDER BY total_flights DESC").show(10, truncate=False)

+-------------------+-----------------+-------------+
|ORIGIN_COUNTRY_NAME|DEST_COUNTRY_NAME|total_flights|
+-------------------+-----------------+-------------+
|United States      |United States    |2119795      |
|Canada             |United States    |49695        |
|United States      |Canada           |49052        |
|Mexico             |United States    |38225        |
|United States      |Mexico           |38075        |
|United States      |United Kingdom   |10946        |
|United Kingdom     |United States    |10358        |
|United States      |Japan            |9205         |
|Japan              |United States    |8643         |
|United States      |Germany          |8501         |
+-------------------+-----------------+-------------+
only showing top 10 rows

